# To check the gpu device

In [15]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


## install YOLOv8

In [16]:

!pip install ultralytics

No global/local python version has been set yet. Please set the global/local version by typing:
pyenv global 3.7.4
pyenv local 3.7.4


#Setting up YOLOv8

In [17]:
from ultralytics import YOLO
import os
from IPython.display import display, Image
from IPython import display
display.clear_output()

# Run YOLO checks using Python instead of shell command
import ultralytics
ultralytics.checks()

Ultralytics 8.4.19  Python-3.12.10 torch-2.10.0+cpu CPU (Intel Core Ultra 5 225H)
Setup complete  (14 CPUs, 31.5 GB RAM, 35.4/238.0 GB disk)


# 📁 Setup Your Custom Dataset

Now we'll configure your local dataset for YOLOv8 training.

## 📋 Dataset Structure Requirements

Your dataset should follow this structure for YOLOv8:

```
your_dataset/
├── train/
│   ├── images/          # Training images (.jpg, .png, .bmp, etc.)
│   │   ├── img1.jpg
│   │   ├── img2.jpg
│   │   └── ...
│   └── labels/          # Training labels (.txt files)
│       ├── img1.txt
│       ├── img2.txt
│       └── ...
├── val/
│   ├── images/          # Validation images
│   └── labels/          # Validation labels
└── test/ (optional)
    ├── images/          # Test images
    └── labels/          # Test labels
```

### 🏷️ Label Format (YOLO format .txt files):
Each image should have a corresponding .txt file with:
```
class_id center_x center_y width height
```
- All coordinates are normalized (0.0 to 1.0)
- One line per object in the image
- Example: `0 0.5 0.3 0.2 0.4` (class 0, centered at 50%,30% with 20%,40% size)

## ✅ Quick Dataset Setup

Since you have the dataset in your project directory, here's what to do:

### 📁 **Current Project Structure:**
```
Traffic-sign-detection-yolov8/
├── TSD_YOLOv8.ipynb           # This notebook
├── dataset/                   # Your dataset folder ← HERE
│   ├── train/
│   │   ├── images/           # Training images
│   │   └── labels/           # Training labels (.txt)
│   ├── val/
│   │   ├── images/           # Validation images  
│   │   └── labels/           # Validation labels (.txt)
│   └── test/ (optional)
│       ├── images/           # Test images
│       └── labels/           # Test labels (.txt)
└── runs/                     # Training results (created automatically)
```

### 🚀 **Ready to start?**
1. **Run cell 10** below to auto-detect your dataset
2. **Run cell 11** to configure paths
3. **Run cell 12** to create YAML config
4. **Run cell 16** to start training!

In [18]:
# Setup Local Dataset - Auto-detect local dataset folder
import os
import shutil

# ===== AUTO-DETECT LOCAL DATASET =====
print("🔍 Looking for dataset in project directory...")

# Check for common dataset folder names
possible_paths = [
    "./DataSet",           # User's folder (capital D & S)
]

# Auto-detect dataset folder
dataset_found = None
for path in possible_paths:
    if os.path.exists(path):
        print(f"✅ Found dataset at: {path}")
        dataset_found = path
        break

if dataset_found:
    print(f"📁 Using auto-detected dataset: {os.path.abspath(dataset_found)}")
    YOUR_DATASET_PATH = os.path.abspath(dataset_found)
    use_auto = input(f"Use auto-detected dataset '{dataset_found}'? (y/n): ").lower()
    
    if use_auto != 'y':
        YOUR_DATASET_PATH = input("Enter full path to your dataset folder: ")
else:
    print("⚠️ No dataset folder auto-detected in project directory")
    print("📝 Expected folder names: dataset, data, datasets, traffic_signs, my_dataset")
    YOUR_DATASET_PATH = input("Enter full path to your dataset folder: ")

print(f"🎯 Selected dataset path: {YOUR_DATASET_PATH}")

# Kiểm tra dataset structure
if os.path.exists(YOUR_DATASET_PATH):
    print("✅ Dataset folder found!")
    
    # List contents
    contents = os.listdir(YOUR_DATASET_PATH)
    print("📁 Dataset contents:")
    for item in contents:
        item_path = os.path.join(YOUR_DATASET_PATH, item)
        if os.path.isdir(item_path):
            # Count files in subdirectories
            try:
                file_count = len(os.listdir(item_path))
                print(f"  📂 {item}/ ({file_count} files)")
            except:
                print(f"  📂 {item}/")
        else:
            print(f"  📄 {item}")
        
    # Check for expected folders
    expected_folders = ['train', 'val', 'test']
    for folder in expected_folders:
        if folder in contents:
            folder_path = os.path.join(YOUR_DATASET_PATH, folder)
            if os.path.exists(os.path.join(folder_path, 'images')):
                img_count = len([f for f in os.listdir(os.path.join(folder_path, 'images')) 
                               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
                print(f"✅ Found {folder} folder with {img_count} images")
            else:
                print(f"⚠️ {folder} folder found but no 'images' subfolder")
        else:
            print(f"⚠️ {folder} folder not found")
else:
    print(f"❌ Dataset not found at: {YOUR_DATASET_PATH}")
    print("Please check the path and try again.")

🔍 Looking for dataset in project directory...
✅ Found dataset at: ./DataSet
📁 Using auto-detected dataset: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\DataSet
🎯 Selected dataset path: 
❌ Dataset not found at: 
Please check the path and try again.


In [19]:
# Dataset Path Configuration - No need to copy if already local
print("📂 Dataset Path Configuration")

# Since dataset is already in project directory, use it directly
if YOUR_DATASET_PATH.startswith('.'):
    # Already local relative path
    DATASET_BASE_PATH = os.path.abspath(YOUR_DATASET_PATH)
    print(f"✅ Using local dataset: {DATASET_BASE_PATH}")
    print("📝 No copying needed - dataset is already in project directory")
else:
    # External path - ask if user wants to copy
    WORKING_DIR = "./dataset"
    
    print(f"🔍 Current dataset location: {YOUR_DATASET_PATH}")
    print(f"🎯 Suggested local location: {os.path.abspath(WORKING_DIR)}")
    
    if os.path.exists(WORKING_DIR):
        print("⚠️ Local dataset folder already exists!")
        overwrite = input("Overwrite existing local dataset? (y/n): ").lower()
        if overwrite == 'y':
            copy_choice = 'y'
        else:
            copy_choice = 'n'
    else:
        copy_choice = input("Copy dataset to project directory for faster access? (y/n): ").lower()
    
    if copy_choice == 'y':
        try:
            if os.path.exists(WORKING_DIR):
                shutil.rmtree(WORKING_DIR)
                print("🗑️ Removed existing dataset folder")
            
            print("📋 Copying dataset... This may take a moment...")
            shutil.copytree(YOUR_DATASET_PATH, WORKING_DIR)
            print(f"✅ Dataset copied to {WORKING_DIR}")
            DATASET_BASE_PATH = os.path.abspath(WORKING_DIR)
        except Exception as e:
            print(f"❌ Error copying dataset: {e}")
            print("Will use original path instead.")
            DATASET_BASE_PATH = os.path.abspath(YOUR_DATASET_PATH)
    else:
        print("Using original dataset path...")
        DATASET_BASE_PATH = os.path.abspath(YOUR_DATASET_PATH)

print(f"🎯 Final dataset path: {DATASET_BASE_PATH}")

📂 Dataset Path Configuration
🔍 Current dataset location: 
🎯 Suggested local location: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\dataset
⚠️ Local dataset folder already exists!
Using original dataset path...
🎯 Final dataset path: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8


In [20]:
# Fix Dataset Path and Create YAML Configuration
import os

# 🔧 FIX: Correct the dataset path if it got corrupted 
if DATASET_BASE_PATH.endswith('\\y') or DATASET_BASE_PATH.endswith('/y') or DATASET_BASE_PATH == 'y':
    print("🔧 Fixing corrupted dataset path...")
    # Use the detected DataSet folder
    if os.path.exists('./DataSet'):
        DATASET_BASE_PATH = os.path.abspath('./DataSet')
        print(f"✅ Fixed path: {DATASET_BASE_PATH}")
    else:
        print("❌ DataSet folder not found. Please create it manually.")
        DATASET_BASE_PATH = input("Enter correct path to your DataSet folder: ")

print(f"📂 Using dataset path: {DATASET_BASE_PATH}")

# Set up paths
train_path = os.path.join(DATASET_BASE_PATH, "train", "images")
val_path = os.path.join(DATASET_BASE_PATH, "val", "images")
test_path = os.path.join(DATASET_BASE_PATH, "test", "images") if os.path.exists(os.path.join(DATASET_BASE_PATH, "test")) else val_path

# Verify paths exist
print("🔍 Checking dataset structure:")
for name, path in [("Train", train_path), ("Validation", val_path), ("Test", test_path)]:
    if os.path.exists(path):
        img_count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
        print(f"✅ {name}: {img_count} images found at {path}")
    else:
        print(f"❌ {name}: Path not found at {path}")

# Get class names with validation
print("\n📋 Please enter your class names:")
while True:
    try:
        num_classes = input("Number of classes: ").strip()
        if num_classes == "":
            print("⚠️ Please enter a number (e.g., 3)")
            continue
        num_classes = int(num_classes)
        if num_classes <= 0:
            print("⚠️ Number of classes must be greater than 0")
            continue
        break
    except ValueError:
        print("⚠️ Please enter a valid number")

class_names = []
for i in range(num_classes):
    while True:
        name = input(f"Class {i} name: ").strip()
        if name:
            class_names.append(name)
            break
        else:
            print("⚠️ Class name cannot be empty")

# Create YAML content
yaml_content = f"""# Custom Traffic Sign Dataset Configuration

# Paths (use forward slashes even on Windows)
train: {train_path.replace(os.sep, '/')}
val: {val_path.replace(os.sep, '/')}
test: {test_path.replace(os.sep, '/')}

# Number of classes
nc: {len(class_names)}

# Class names
names: {class_names}
"""

# Write YAML file
try:
    with open('data_custom.yaml', 'w', encoding='utf-8') as f:
        f.write(yaml_content)
    print("✅ data_custom.yaml created successfully!")
    print("\n📄 Content:")
    print(yaml_content)
except Exception as e:
    print(f"❌ Error creating YAML file: {e}")

📂 Using dataset path: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8
🔍 Checking dataset structure:
❌ Train: Path not found at d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\train\images
❌ Validation: Path not found at d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\val\images
❌ Test: Path not found at d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\val\images

📋 Please enter your class names:
✅ data_custom.yaml created successfully!

📄 Content:
# Custom Traffic Sign Dataset Configuration

# Paths (use forward slashes even on Windows)
train: d:/Detect_traffic_sign/Traffic-sign-detection-yolov8/train/images
val: d:/Detect_traffic_sign/Traffic-sign-detection-yolov8/val/images
test: d:/Detect_traffic_sign/Traffic-sign-detection-yolov8/val/images

# Number of classes
nc: 1

# Class names
names: ['1']



In [21]:
# 🔍 Auto-detect number of classes from dataset
import os
import glob

print("🔍 Analyzing your dataset to detect number of classes...")

# Fix dataset path if corrupted
if 'DATASET_BASE_PATH' in locals():
    if DATASET_BASE_PATH.endswith('\\y') or DATASET_BASE_PATH.endswith('/y') or DATASET_BASE_PATH == 'y':
        if os.path.exists('./DataSet'):
            DATASET_BASE_PATH = os.path.abspath('./DataSet')
            print(f"🔧 Fixed dataset path: {DATASET_BASE_PATH}")
        else:
            DATASET_BASE_PATH = input("📁 Enter path to your DataSet folder: ")
else:
    # If DATASET_BASE_PATH not defined, detect it
    if os.path.exists('./DataSet'):
        DATASET_BASE_PATH = os.path.abspath('./DataSet')
        print(f"📂 Found dataset at: {DATASET_BASE_PATH}")
    else:
        DATASET_BASE_PATH = input("📁 Enter path to your DataSet folder: ")

# Check label folders
label_folders = [
    os.path.join(DATASET_BASE_PATH, "train", "labels"),
    os.path.join(DATASET_BASE_PATH, "val", "labels")
]

class_ids = set()
total_labels = 0

for label_folder in label_folders:
    if os.path.exists(label_folder):
        print(f"📋 Checking labels in: {label_folder}")
        
        # Find all .txt files
        txt_files = glob.glob(os.path.join(label_folder, "*.txt"))
        print(f"   Found {len(txt_files)} label files")
        
        # Read each .txt file to extract class IDs
        for txt_file in txt_files[:10]:  # Check first 10 files for speed
            try:
                with open(txt_file, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line:  # Skip empty lines
                            parts = line.split()
                            if len(parts) >= 5:  # Valid YOLO format: class_id x y w h
                                class_id = int(parts[0])
                                class_ids.add(class_id)
                                total_labels += 1
            except Exception as e:
                print(f"   ⚠️ Could not read {txt_file}: {e}")
    else:
        print(f"❌ Label folder not found: {label_folder}")

# Results
if class_ids:
    sorted_classes = sorted(class_ids)
    num_classes = len(sorted_classes)
    
    print(f"\n🎯 DETECTION RESULTS:")
    print(f"✅ Found {num_classes} classes in your dataset")
    print(f"📊 Class IDs found: {sorted_classes}")
    print(f"📝 Total labels analyzed: {total_labels}")
    
    print(f"\n💡 You should enter: {num_classes}")
    
    # Auto-suggest class names based on common traffic signs
    suggested_names = []
    common_traffic_signs = [
        "stop_sign", "speed_limit", "no_entry", "warning", "yield", 
        "turn_left", "turn_right", "no_parking", "school_zone", "construction"
    ]
    
    for i in range(num_classes):
        if i < len(common_traffic_signs):
            suggested_names.append(common_traffic_signs[i])
        else:
            suggested_names.append(f"class_{i}")
    
    print(f"💭 Suggested class names: {suggested_names}")
    
else:
    print("\n❌ No valid class IDs found in label files!")
    print("🔍 Please check:")
    print("   - Do you have .txt files in DataSet/train/labels/ ?")
    print("   - Are the .txt files in YOLO format (class_id x y w h) ?")
    
    # Show sample label files if exist
    for label_folder in label_folders:
        if os.path.exists(label_folder):
            sample_files = glob.glob(os.path.join(label_folder, "*.txt"))[:3]
            if sample_files:
                print(f"\n📄 Sample from {label_folder}:")
                for sample_file in sample_files:
                    print(f"   📃 {os.path.basename(sample_file)}")
                    try:
                        with open(sample_file, 'r') as f:
                            lines = f.readlines()[:3]
                            for line in lines:
                                print(f"      {line.strip()}")
                    except:
                        pass

🔍 Analyzing your dataset to detect number of classes...
❌ Label folder not found: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\train\labels
❌ Label folder not found: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\val\labels

❌ No valid class IDs found in label files!
🔍 Please check:
   - Do you have .txt files in DataSet/train/labels/ ?
   - Are the .txt files in YOLO format (class_id x y w h) ?


In [22]:
# ✅ Update Dataset Configuration for Train + Val Only
import os

print("📂 Cập nhật cấu hình dataset cho cấu trúc Train + Val")

# Use existing DATASET_BASE_PATH if available
if 'DATASET_BASE_PATH' in locals() and DATASET_BASE_PATH:
    print(f"📁 Sử dụng dataset path hiện tại: {DATASET_BASE_PATH}")
else:
    # Auto-detect DataSet folder
    if os.path.exists('./DataSet'):
        DATASET_BASE_PATH = os.path.abspath('./DataSet')
        print(f"🔍 Tự động phát hiện: {DATASET_BASE_PATH}")
    else:
        DATASET_BASE_PATH = input("📁 Nhập đường dẫn tới folder DataSet: ")

# ✅ Cấu trúc dataset chỉ có train và val
train_images = os.path.join(DATASET_BASE_PATH, "train", "images")
train_labels = os.path.join(DATASET_BASE_PATH, "train", "labels")
val_images = os.path.join(DATASET_BASE_PATH, "val", "images")  
val_labels = os.path.join(DATASET_BASE_PATH, "val", "labels")

print("\n🔍 Kiểm tra cấu trúc dataset (Train + Val only):")

# Kiểm tra từng folder
folders_to_check = [
    ("Train Images", train_images),
    ("Train Labels", train_labels), 
    ("Val Images", val_images),
    ("Val Labels", val_labels)
]

dataset_valid = True
for name, path in folders_to_check:
    if os.path.exists(path):
        file_count = len([f for f in os.listdir(path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.txt'))])
        print(f"✅ {name}: {file_count} files - {path}")
    else:
        print(f"❌ {name}: Không tìm thấy - {path}")
        dataset_valid = False

if dataset_valid:
    print("\n🎯 Dataset structure hợp lệ!")
    
    # Sử dụng class info đã có hoặc auto-detect
    if 'class_names' in locals() and class_names:
        print(f"📋 Sử dụng class names hiện tại: {class_names}")
    else:
        print("🔍 Tự động phát hiện classes...")
        # Simple auto-detection from previous analysis
        if 'sorted_classes' in locals() and sorted_classes:
            num_classes = len(sorted_classes)
            class_names = [f"traffic_sign_{i}" for i in range(num_classes)]
            print(f"📊 Phát hiện {num_classes} classes: {class_names}")
        else:
            # Manual input nếu cần
            num_classes = int(input("Số lượng classes: ") or "1")
            class_names = []
            for i in range(num_classes):
                name = input(f"Tên class {i}: ") or f"class_{i}"
                class_names.append(name)
    
    # ✅ Tạo YAML cho dataset Train + Val (không có Test)
    yaml_content = f"""# Custom Traffic Sign Dataset - Train + Val Only

# Dataset paths  
train: {train_images.replace(os.sep, '/')}
val: {val_images.replace(os.sep, '/')}
# test: {val_images.replace(os.sep, '/')}  # Use val for testing

# Classes
nc: {len(class_names)}
names: {class_names}
"""

    # Ghi file YAML
    try:
        with open('data_custom.yaml', 'w', encoding='utf-8') as f:
            f.write(yaml_content)
        print("\n✅ File data_custom.yaml đã được cập nhật!")
        print("📄 Nội dung:")
        print(yaml_content)
        
        # Hiển thị thống kê dataset
        train_img_count = len([f for f in os.listdir(train_images) 
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
        val_img_count = len([f for f in os.listdir(val_images) 
                            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
        
        print(f"\n📊 Thống kê dataset:")
        print(f"   🏋️ Train: {train_img_count} images")  
        print(f"   🎯 Val: {val_img_count} images")
        print(f"   📁 Classes: {len(class_names)}")
        
    except Exception as e:
        print(f"❌ Lỗi tạo YAML file: {e}")
        
else:
    print("\n⚠️ Cấu trúc dataset không hợp lệ!")
    print("💡 Dataset cần có cấu trúc:")
    print("   DataSet/")
    print("   ├── train/")
    print("   │   ├── images/")
    print("   │   └── labels/") 
    print("   └── val/")
    print("       ├── images/")
    print("       └── labels/")

📂 Cập nhật cấu hình dataset cho cấu trúc Train + Val
📁 Sử dụng dataset path hiện tại: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8

🔍 Kiểm tra cấu trúc dataset (Train + Val only):
❌ Train Images: Không tìm thấy - d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\train\images
❌ Train Labels: Không tìm thấy - d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\train\labels
❌ Val Images: Không tìm thấy - d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\val\images
❌ Val Labels: Không tìm thấy - d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\val\labels

⚠️ Cấu trúc dataset không hợp lệ!
💡 Dataset cần có cấu trúc:
   DataSet/
   ├── train/
   │   ├── images/
   │   └── labels/
   └── val/
       ├── images/
       └── labels/


In [23]:
# ✅ Verify New Dataset và Create Final YAML
import os
import glob

print("🔍 Kiểm tra dataset mới đã thêm...")

# Set correct dataset path
DATASET_BASE_PATH = os.path.abspath("./DataSet")
print(f"📂 Dataset path: {DATASET_BASE_PATH}")

# Check dataset structure
train_images = os.path.join(DATASET_BASE_PATH, "train", "images")
train_labels = os.path.join(DATASET_BASE_PATH, "train", "labels")
val_images = os.path.join(DATASET_BASE_PATH, "val", "images")
val_labels = os.path.join(DATASET_BASE_PATH, "val", "labels")

print("\n📊 Thống kê dataset hiện tại:")

# Count files
for name, path in [("Train Images", train_images), ("Train Labels", train_labels), 
                   ("Val Images", val_images), ("Val Labels", val_labels)]:
    if os.path.exists(path):
        if "images" in name.lower():
            count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
        else:
            count = len([f for f in os.listdir(path) if f.lower().endswith('.txt')])
        print(f"✅ {name}: {count} files")
    else:
        print(f"❌ {name}: Không tồn tại")

# Auto-detect classes from labels
print("\n🔍 Phân tích classes trong dataset...")
class_ids = set()
total_annotations = 0

label_folders = [train_labels, val_labels]
for label_folder in label_folders:
    if os.path.exists(label_folder):
        txt_files = glob.glob(os.path.join(label_folder, "*.txt"))
        folder_name = os.path.basename(label_folder)
        print(f"   📋 {folder_name}: {len(txt_files)} label files")
        
        for txt_file in txt_files[:20]:  # Check first 20 files
            try:
                with open(txt_file, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            parts = line.split()
                            if len(parts) >= 5:
                                class_id = int(parts[0])
                                class_ids.add(class_id)
                                total_annotations += 1
            except Exception as e:
                print(f"   ⚠️ Lỗi đọc {os.path.basename(txt_file)}: {e}")

if class_ids:
    sorted_classes = sorted(class_ids)
    num_classes = len(sorted_classes)
    
    print(f"\n🎯 KẾT QUẢ PHÂN TÍCH:")
    print(f"✅ Tìm thấy {num_classes} classes")
    print(f"📊 Class IDs: {sorted_classes}")
    print(f"📝 Tổng annotations: {total_annotations}")
    
    # Create class names
    class_names = []
    for i in range(num_classes):
        class_names.append(f"traffic_sign_{sorted_classes[i]}")
    
    print(f"🏷️ Class names: {class_names}")
    
    # Create YAML configuration
    yaml_content = f"""# Traffic Sign Dataset Configuration
# Auto-generated for Train + Val dataset

# Dataset paths
train: {train_images.replace(os.sep, '/')}
val: {val_images.replace(os.sep, '/')}

# Classes
nc: {num_classes}
names: {class_names}
"""

    # Write YAML file
    try:
        with open('data_custom.yaml', 'w', encoding='utf-8') as f:
            f.write(yaml_content)
        print(f"\n✅ File 'data_custom.yaml' đã được tạo thành công!")
        print("\n📄 Nội dung YAML:")
        print(yaml_content)
        
        # Double check
        if os.path.exists('data_custom.yaml'):
            print("🔍 Xác nhận: data_custom.yaml tồn tại và sẵn sàng cho training!")
        else:
            print("❌ Lỗi: Không tạo được file YAML")
            
    except Exception as e:
        print(f"❌ Lỗi tạo YAML file: {e}")

else:
    print("\n❌ KHÔNG TÌM THẤY CLASS NÀO!")
    print("🔍 Kiểm tra:")
    print("   - Các file .txt có đúng format YOLO không?")
    print("   - Format: class_id center_x center_y width height")
    
    # Show sample files
    for label_folder in [train_labels, val_labels]:
        if os.path.exists(label_folder):
            sample_files = glob.glob(os.path.join(label_folder, "*.txt"))[:2]
            if sample_files:
                print(f"\n📄 Sample từ {os.path.basename(label_folder)}:")
                for sample in sample_files:
                    print(f"   📃 {os.path.basename(sample)}:")
                    try:
                        with open(sample, 'r') as f:
                            lines = f.readlines()[:3]
                            for line in lines:
                                print(f"      {line.strip()}")
                    except:
                        print("      (không đọc được)")
                        
print("\n🎯 Dataset verification completed!")

🔍 Kiểm tra dataset mới đã thêm...
📂 Dataset path: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\DataSet

📊 Thống kê dataset hiện tại:
✅ Train Images: 82 files
✅ Train Labels: 82 files
✅ Val Images: 19 files
✅ Val Labels: 19 files

🔍 Phân tích classes trong dataset...
   📋 labels: 82 label files
   📋 labels: 19 label files

🎯 KẾT QUẢ PHÂN TÍCH:
✅ Tìm thấy 1 classes
📊 Class IDs: [15]
📝 Tổng annotations: 42
🏷️ Class names: ['traffic_sign_15']

✅ File 'data_custom.yaml' đã được tạo thành công!

📄 Nội dung YAML:
# Traffic Sign Dataset Configuration
# Auto-generated for Train + Val dataset

# Dataset paths
train: d:/Detect_traffic_sign/Traffic-sign-detection-yolov8/DataSet/train/images
val: d:/Detect_traffic_sign/Traffic-sign-detection-yolov8/DataSet/val/images

# Classes
nc: 1
names: ['traffic_sign_15']

🔍 Xác nhận: data_custom.yaml tồn tại và sẵn sàng cho training!

🎯 Dataset verification completed!


# Train YOLOv8 Model on Custom Dataset

## 📋 Configuration Complete!

The **data_custom.yaml** file will be automatically created with your dataset paths and class names.

## 💡 Example YAML Configuration

Your **data_custom.yaml** will look something like this:

```yaml
# Custom Traffic Sign Dataset Configuration

# Paths (automatically adjusted for your system)
train: D:/Detect_traffic_sign/Traffic-sign-detection-yolov8/dataset/train/images
val: D:/Detect_traffic_sign/Traffic-sign-detection-yolov8/dataset/val/images
test: D:/Detect_traffic_sign/Traffic-sign-detection-yolov8/dataset/val/images

# Number of classes (you will specify this)
nc: 3

# Class names (you will enter these)
names: ['stop_sign', 'speed_limit', 'warning']
```

**Don't worry** - this will be generated automatically in the next step! 🎯

In [24]:
# 🚀 Train YOLOv8 Model on Custom Dataset - Fixed Python method
import os
from ultralytics import YOLO

if os.path.exists('data_custom.yaml'):
    print("✅ Starting training with custom dataset...")
    print("⏰ This may take a while depending on your dataset size and hardware.\n")
    
    try:
        # Initialize model
        model = YOLO('yolov8m.pt')
        
        # Start training using Python API instead of shell command
        results = model.train(
            data='data_custom.yaml',
            epochs=20,           # Reduced epochs for testing
            imgsz=300,
            batch=2,
            patience=10,
            save_period=5
        )
        
        print("\n🎉 Training completed! Check runs/detect/train/ for results.")
        
    except Exception as e:
        print(f"❌ Training error: {e}")
        print("Trying alternative method...")
        
        # Alternative: Use os.system
        train_cmd = "yolo task=detect mode=train model=yolov8m.pt data=data_custom.yaml epochs=20 imgsz=300 batch=2"
        print(f"Running: {train_cmd}")
        result = os.system(train_cmd)
        
        if result == 0:
            print("✅ Training completed!")
        else:
            print(f"❌ Training failed with exit code: {result}")
            
else:
    print("❌ data_custom.yaml not found!")
    print("Available YAML files:")
    for file in os.listdir('.'):
        if file.endswith('.yaml'):
            print(f"  - {file}")
    print("Please run the dataset setup cells first.")

✅ Starting training with custom dataset...
⏰ This may take a while depending on your dataset size and hardware.

Ultralytics 8.4.19  Python-3.12.10 torch-2.10.0+cpu CPU (Intel Core Ultra 5 225H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_custom.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=300, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train6, nbs=64, n

In [25]:
# Check if confusion matrix exists and display it
import os
confusion_matrix_path = 'runs/detect/train/confusion_matrix.png'

if os.path.exists(confusion_matrix_path):
    Image(filename=confusion_matrix_path, width=600)
else:
    print("⚠️  Confusion matrix not found!")
    print(f"Expected path: {os.path.abspath(confusion_matrix_path)}")
    print("Make sure you have run the training cell first and it completed successfully.")
    
    # Check if runs folder exists at all
    if os.path.exists('runs'):
        print(f"✅ 'runs' folder exists")
        # List contents of runs folder
        for root, dirs, files in os.walk('runs'):
            level = root.replace('runs', '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            subindent = ' ' * 2 * (level + 1)
            for file in files:
                print(f"{subindent}{file}")
    else:
        print("❌ 'runs' folder does not exist - training hasn't been run yet")

⚠️  Confusion matrix not found!
Expected path: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\runs\detect\train\confusion_matrix.png
Make sure you have run the training cell first and it completed successfully.
✅ 'runs' folder exists
runs/
  detect/
    train/
      args.yaml
      weights/
    train2/
      args.yaml
      weights/
    train3/
      args.yaml
      weights/
    train4/
      args.yaml
      weights/
    train5/
      args.yaml
      weights/
    train6/
      args.yaml
      weights/


In [26]:
# 📊 Validate the trained model
import os

best_model_path = 'runs/detect/train/weights/best.pt'
yaml_config = 'data_custom.yaml'

if os.path.exists(best_model_path) and os.path.exists(yaml_config):
    print("✅ Running model validation...")
    !yolo task=detect mode=val model=runs/detect/train/weights/best.pt data=data_custom.yaml
    print("📈 Validation completed!")
else:
    if not os.path.exists(best_model_path):
        print(f"❌ Model not found at {best_model_path}")
        print("Please run the training cell first and wait for it to complete.")
    if not os.path.exists(yaml_config):
        print(f"❌ {yaml_config} missing")
        print("Please run the dataset setup cells first.")
        
    print("\n💡 Training creates the following files:")
    print("  - runs/detect/train/weights/best.pt (best model)")
    print("  - runs/detect/train/weights/last.pt (last epoch)")
    print("  - runs/detect/train/results.csv (training metrics)")

❌ Model not found at runs/detect/train/weights/best.pt
Please run the training cell first and wait for it to complete.

💡 Training creates the following files:
  - runs/detect/train/weights/best.pt (best model)
  - runs/detect/train/weights/last.pt (last epoch)
  - runs/detect/train/results.csv (training metrics)


In [27]:
# 🔮 Run predictions on validation set
import os

best_model_path = 'runs/detect/train/weights/best.pt'

# Get validation path from YAML config
source_path = None
if os.path.exists('data_custom.yaml'):
    with open('data_custom.yaml', 'r') as f:
        for line in f:
            if line.startswith('val:'):
                source_path = line.split(':', 1)[1].strip()
                break
    
    # Convert back to OS-specific path
    if source_path:
        source_path = source_path.replace('/', os.sep)

if not source_path:
    source_path = 'val/images'

print(f"🎯 Prediction source: {source_path}")

if os.path.exists(best_model_path) and os.path.exists(source_path):
    print("✅ Running predictions...")
    !yolo task=detect mode=predict model=runs/detect/train/weights/best.pt save=True conf=0.2 source="{source_path}"
    print("🖼️ Predictions saved to runs/detect/predict/")
else:
    if not os.path.exists(best_model_path):
        print(f"❌ Model not found at {best_model_path}")
    if not os.path.exists(source_path):
        print(f"❌ Images not found at {source_path}")
    print("Please ensure training is complete and dataset is properly configured.")

🎯 Prediction source: d:\Detect_traffic_sign\Traffic-sign-detection-yolov8\DataSet\val\images
❌ Model not found at runs/detect/train/weights/best.pt
Please ensure training is complete and dataset is properly configured.


In [28]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg'):
    display(Image(filename=image_path, height=300))
    print('\n')

# 🎯 Trained Model Location

After training completes successfully, you can find your trained model here:

**Windows Path:** `runs\detect\train\weights\best.pt`

**Alternative models:**
- `runs\detect\train\weights\last.pt` - Model from the last epoch
- `runs\detect\train\weights\best.pt` - Best performing model

## 📊 Training Results
You can also find these useful files:
- `runs\detect\train\results.csv` - Training metrics and loss curves
- `runs\detect\train\confusion_matrix.png` - Confusion matrix
- `runs\detect\train\results.png` - Training charts
- `runs\detect\train\val_batch0_labels.jpg` - Validation samples with ground truth
- `runs\detect\train\val_batch0_pred.jpg` - Validation samples with predictions

## 🚀 Using the Trained Model
```python
from ultralytics import YOLO

# Load your trained model
model = YOLO('runs/detect/train/weights/best.pt')

# Make predictions
results = model('path/to/your/image.jpg')
results[0].show()  # Display results
```